# P81 — Introducción a la selección de variables y características

## 1. Título y paper

**Paper:** *An Introduction to Variable and Feature Selection*  
**Autoría:** Isabelle Guyon, André Elisseeff  
**Año y venue:** 2003 · Journal of Machine Learning Research, 3, 1157–1182  
**Nivel:** L3 · **Motor:** `seleccion_de_caracteristicas`  
**Ficha completa:** [`P81_seleccion_de_caracteristicas`](../../papers/foundational/P81_seleccion_de_caracteristicas/README.md)

**Hito:** Ordena el problema de elegir variables y demuestra por qué el ranking de una en una falla en las dos direcciones.

- [JMLR 3:1157–1182](https://www.jmlr.org/papers/v3/guyon03a.html)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Con miles de variables y pocas muestras hay que reducir. El método habitual —ordenar las variables por su correlación con la etiqueta y quedarse con las primeras— tiene modos de fallo que casi nadie enunciaba.
2. Ejecutar una implementación mínima de la propuesta: Un marco con tres familias —filtros, envolturas y métodos embebidos— y dos advertencias con contraejemplo: una variable inútil por separado puede ser imprescindible en compañía, y dos variables redundantes pueden ser mejores juntas que cualquiera sola.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P77
- Kohavi y John (1997), envolturas


## 4. Intuición

Ordenar las variables por su correlación con la etiqueta y quedarse con las mejores parece sensato. Falla en las dos direcciones: descarta variables inútiles solas e imprescindibles juntas, y descarta variables «redundantes» que juntas cancelan ruido.


## 5. Concepto mínimo

```text
Caso 1 — complementariedad:
    a, b ∈ {−1, +1},  y = 1 si a·b > 0
    correlación(a, y) ≈ 0   correlación(b, y) ≈ 0   y JUNTAS lo determinan

Caso 2 — redundancia útil:
    r1 = señal + ruido₁,  r2 = señal + ruido₂
    promediarlas reduce la varianza del ruido a la mitad
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('seleccion_de_caracteristicas', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto acierta un clasificador con «a» sola? ¿Y con «b» sola?
2. ¿Y con las dos?
3. ¿Mejora promediar dos variables que miden lo mismo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('seleccion_de_caracteristicas', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('seleccion_de_caracteristicas', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con «a» sola se acierta **0,5** y con «b» sola, **0,5083**: el azar. Con las dos, **1,0**. Y sus correlaciones univariantes con la etiqueta son 0,09 y 0,009 — un ranking las descartaría antes de llegar a mirarlas juntas. En sentido contrario, promediar «r1» y «r2» baja el error de estimación de 0,5379 a 0,3003.


## 10. Comentario pedagógico

Los dos casos tienen la misma consecuencia práctica: el ranking univariante es un método con modos de fallo conocidos, y conviene usarlo sabiéndolos. El artículo ordena las alternativas —filtros, envolturas, métodos embebidos— y el lasso de [P77](../../papers/foundational/P77_lasso/README.md) es un ejemplo de la tercera familia.


## 11. Error o anti-patrón deliberado

Anti-patrón: eliminar variables correlacionadas entre sí «porque son redundantes».


In [ ]:
print('Dos variables que miden lo mismo con ruido independiente NO sobran:')
print('promediarlas reduce la varianza del ruido.')
print('«Redundante» es una propiedad de la informacion, no de la correlacion.')

## 12. Corrección

La comprobación numérica, sobre las mismas variables:


In [ ]:
r = run_paper_lab('seleccion_de_caracteristicas', seed=7)['result']
print('complementariedad:', r['caso_1_variables_inutiles_por_separado'])
print('redundancia util :', r['caso_2_variables_redundantes'])

## 13. Desafío guiado

Revisa el ranking univariante de la salida y señala en qué posición quedarían «a» y «b», las dos variables que determinan por completo la etiqueta.


In [ ]:
r = run_paper_lab('seleccion_de_caracteristicas', seed=3)['result']
show(r)

## 14. Desafío autónomo

Toma un conjunto propio y compara tres estrategias: ranking univariante, selección hacia delante con envoltura, y lasso. Documenta cuántas variables selecciona cada una y si coinciden.


## 15. Evidencia de aprendizaje

Guarda los dos contraejemplos con sus números y tu enunciado de los dos modos de fallo del ranking univariante.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P81_seleccion_de_caracteristicas/README.md) · evaluación formal: [`assessments/papers/P81_seleccion_de_caracteristicas.md`](../../assessments/papers/P81_seleccion_de_caracteristicas.md)


## 16. Cierre

Ya sabemos qué darle al modelo. Falta comprobar que el número que devuelve significa lo que parece significar.


## 17. Conexión con el siguiente hito

- P52

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
